# Aleppo Historic Network Distance Analysis

This notebook supports a graduate historic preservation research assignment focused on the Ancient City of Aleppo as a networked urban fabric.

## 1. Research Statement

> This study models the historic urban fabric of the Ancient City of Aleppo (UNESCO World Heritage Site) as a network, using the medieval souk and street system as edges and conflict-damaged heritage buildings (2012–2016 Syrian Civil War) as nodes. It compares Euclidean (straight-line) distance against network (path-based, along real streets) distance between damaged heritage sites, in order to examine how the experience of proximity in the historic city differs from proximity as measured on a flat map — and how the destruction of specific souk corridors or landmark buildings disrupted a fabric that was previously legible as a walkable, continuous whole.

## 2. Data Sources

This notebook uses three shapefile bundles already present in the working directory:

- 6_Damage_Sites_Aleppo_SDA.shp — UNOSAT point-level conflict damage assessment (event code CE20130604SYR, Syrian Civil War, 2012–2016), 35,936 points citywide. Key columns: SiteID, DmgCls_4, SensDt_4, GrpDmgCls, FldValid, Notes, Neighbrhd, StlmtNme, EventCode. CRS: EPSG:4326.
- 7_Aleppo_PercentageDamage_Neighborhood.shp — 127 named neighborhood/district polygons with a Percent field, Name and Name_Ar. CRS: EPSG:4326. Includes Old City quarters such as Aleppo Citadel, Al Jalloum, al Aqabeh, Farafira, Bayadah, al Kallasah, As-Sukkari, Bab Alfaraj, and others.
- 8_Aleppo_ResidentialPercentageDamage.shp — 553,296-polygon rasterized damage-density surface (ID, GRIDCODE 0–100). Not used for node/edge construction; it is included only as an optional visual backdrop.

## 3. Load and Filter Damage Sites

This code loads the UNOSAT damage points, keeps only the named sites that can act as meaningful landmarks, and prepares a cleaned node list for network analysis.

In [1]:
import warnings
import os
import pandas as pd
import networkx as nx
import geopandas as gpd
from pathlib import Path

warnings.filterwarnings("ignore")

# Working directory and data paths
DATA_DIR = Path.cwd()
DATA_CANDIDATES = [
    Path(
        "/Users/khaledalanjery/Library/CloudStorage/GoogleDrive-khaled@khaledalanjery.com/My Drive/Design Independent Research and Experimentation/DIRE Projects/Aeolian/Aeolian General/Aeolian Github/aeolian-project-repo/msys-project-repo-folder/damage/damage data/damage sites aleppo"
    ),
    DATA_DIR
    / "content/assignments/mapping-systems-assignments/damage/damage data/UNOSAT_CE20130604SYR_Syria_Damage_Assessment_2016_shp",
    DATA_DIR
    / "content/assignments/mapping-systems-assignments/damage/damage data/damage sites aleppo",
    DATA_DIR,
]


def find_data_file(name):
    for base in DATA_CANDIDATES:
        candidate = Path(base) / name
        if candidate.exists():
            return candidate
    return None


damage_points_path = find_data_file("6_Damage_Sites_Aleppo_SDA.shp")
neighborhood_path = find_data_file("7_Aleppo_PercentageDamage_Neighborhood.shp")
residential_damage_path = find_data_file("8_Aleppo_ResidentialPercentageDamage.shp")

# Load source data in EPSG:4326
if damage_points_path is not None:
    gdf_sites = gpd.read_file(damage_points_path)
else:
    gdf_sites = gpd.GeoDataFrame(
        columns=["SiteID", "Notes", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

if neighborhood_path is not None:
    gdf_neighborhoods = gpd.read_file(neighborhood_path)
else:
    gdf_neighborhoods = gpd.GeoDataFrame(
        columns=["Name", "Percent", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

if residential_damage_path is not None:
    gdf_residential = gpd.read_file(residential_damage_path)
else:
    gdf_residential = gpd.GeoDataFrame(
        columns=["ID", "GRIDCODE", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

# Filter to Aleppo historic-district neighborhoods and sites
historic_keywords = [
    "citadel",
    "jalloum",
    "aqabeh",
    "farafira",
    "bayadah",
    "kallasah",
    "sukkari",
    "bab alfaraj",
    "old city",
    "souk",
    "historic district",
]

name_columns = [
    col
    for col in ["Name", "Name_Ar", "Neighbrhd", "StlmtNme"]
    if col in gdf_neighborhoods.columns
]
if len(gdf_neighborhoods) > 0 and name_columns:
    historic_mask = pd.Series(False, index=gdf_neighborhoods.index)
    for col in name_columns:
        values = gdf_neighborhoods[col].fillna("").astype(str).str.lower()
        historic_mask = historic_mask | values.str.contains(
            "|".join(historic_keywords), na=False
        )
    gdf_historic_district = gdf_neighborhoods.loc[historic_mask].copy()
else:
    gdf_historic_district = gdf_neighborhoods.copy()

if (
    len(gdf_historic_district) > 0
    and len(gdf_sites) > 0
    and "geometry" in gdf_sites.columns
    and "geometry" in gdf_historic_district.columns
):
    gdf_sites = gpd.sjoin(
        gdf_sites.to_crs(gdf_historic_district.crs),
        gdf_historic_district[["Name", "Name_Ar", "geometry"]],
        how="inner",
        predicate="within",
    )
    if "index_right" in gdf_sites.columns:
        gdf_sites = gdf_sites.drop(columns=["index_right"])
else:
    site_text_columns = [
        col
        for col in ["Notes", "StlmtNme", "Neighbrhd", "SiteID"]
        if col in gdf_sites.columns
    ]
    if len(gdf_sites) > 0 and site_text_columns:
        combined = gdf_sites[site_text_columns].fillna("").astype(str)
        site_mask = combined.apply(
            lambda row: any(
                keyword in " ".join(row.tolist()).lower()
                for keyword in historic_keywords
            ),
            axis=1,
        )
        gdf_sites = gdf_sites.loc[site_mask].copy()

if (
    len(gdf_historic_district) > 0
    and len(gdf_residential) > 0
    and "geometry" in gdf_residential.columns
):
    gdf_residential = gpd.sjoin(
        gdf_residential.to_crs(gdf_historic_district.crs),
        gdf_historic_district[["geometry"]],
        how="inner",
        predicate="intersects",
    )
    if "index_right" in gdf_residential.columns:
        gdf_residential = gdf_residential.drop(columns=["index_right"])

print(
    "Loaded",
    len(gdf_sites),
    "damage points,",
    len(gdf_historic_district),
    "historic-district neighborhoods, and",
    len(gdf_residential),
    "residential damage features",
)

Loaded 2879 damage points, 8 historic-district neighborhoods, and 20459 residential damage features


In [15]:
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

MAPBOX_TOKEN = "pk.eyJ1Ijoia2hhbGVkYWxhbmplcnkiLCJhIjoiY21zOXJtdGE3MHM5NDJ3b2RhZDhlN3RzdiJ9.u-954HDL1cS8ZVODaKr3IQ"

# Load neighborhood boundaries if they were not already created earlier in the notebook
if "gdf_historic_district" not in globals():
    DATA_DIR = Path.cwd()
    DATA_CANDIDATES = [
        Path(
            "/Users/khaledalanjery/Library/CloudStorage/GoogleDrive-khaled@khaledalanjery.com/My Drive/Design Independent Research and Experimentation/DIRE Projects/Aeolian/Aeolian General/Aeolian Github/aeolian-project-repo/msys-project-repo-folder/damage/damage data/damage sites aleppo"
        ),
        DATA_DIR
        / "content/assignments/mapping-systems-assignments/damage/damage data/UNOSAT_CE20130604SYR_Syria_Damage_Assessment_2016_shp",
        DATA_DIR
        / "content/assignments/mapping-systems-assignments/damage/damage data/damage sites aleppo",
        DATA_DIR,
    ]

    def find_data_file(name):
        for base in DATA_CANDIDATES:
            candidate = Path(base) / name
            if candidate.exists():
                return candidate
        return None

    neighborhood_path = find_data_file("7_Aleppo_PercentageDamage_Neighborhood.shp")
    if neighborhood_path is not None:
        gdf_neighborhoods = gpd.read_file(neighborhood_path)
        historic_keywords = [
            "citadel",
            "jalloum",
            "aqabeh",
            "farafira",
            "bayadah",
            "kallasah",
            "sukkari",
            "bab alfaraj",
            "old city",
            "souk",
            "historic district",
        ]
        name_columns = [
            col
            for col in ["Name", "Name_Ar", "Neighbrhd", "StlmtNme"]
            if col in gdf_neighborhoods.columns
        ]
        if len(gdf_neighborhoods) > 0 and name_columns:
            historic_mask = pd.Series(False, index=gdf_neighborhoods.index)
            for col in name_columns:
                values = gdf_neighborhoods[col].fillna("").astype(str).str.lower()
                historic_mask = historic_mask | values.str.contains(
                    "|".join(historic_keywords), na=False
                )
            gdf_historic_district = gdf_neighborhoods.loc[historic_mask].copy()
        else:
            gdf_historic_district = gdf_neighborhoods.copy()
    else:
        gdf_historic_district = gpd.GeoDataFrame(
            columns=["Name", "geometry"], geometry="geometry", crs="EPSG:4326"
        )

# Load the Aleppo damage points if they were not already created earlier in the notebook
if "gdf_sites" not in globals():
    aleppo_path = Path(
        "/Users/khaledalanjery/Library/CloudStorage/GoogleDrive-khaled@khaledalanjery.com/My Drive/Design Independent Research and Experimentation/DIRE Projects/Aeolian/Aeolian General/Aeolian Github/aeolian-project-repo/msys-project-repo-folder/damage/damage data/damage sites aleppo/6_Damage_Sites_Aleppo_SDA.shp"
    )
    if aleppo_path.exists():
        gdf_sites = gpd.read_file(aleppo_path)
    else:
        gdf_sites = gpd.GeoDataFrame(
            columns=["DmgCls_4", "geometry"], geometry="geometry", crs="EPSG:4326"
        )

# Keep only rows that have a usable geometry and damage class
plot_gdf = gdf_sites.dropna(subset=["geometry"]).copy()
if "DmgCls_4" in plot_gdf.columns:
    plot_gdf = plot_gdf.dropna(subset=["DmgCls_4"]).copy()
if getattr(plot_gdf, "crs", None) is not None:
    plot_gdf = plot_gdf.to_crs(epsg=4326)
plot_gdf["lon"] = plot_gdf.geometry.x
plot_gdf["lat"] = plot_gdf.geometry.y
plot_gdf = plot_gdf.dropna(subset=["lon", "lat"])

# Sample for a responsive map while preserving the overall pattern
sample_size = min(1500, len(plot_gdf))
plot_gdf = plot_gdf.sample(n=sample_size, random_state=42).copy()

# Color points by damage class using the requested palette
color_map = {
    "Destroyed": "#ff0000",
    "Severe Damage": "#ff8c00",
    "Moderate Damage": "#ffffff",
    "Medium Moderate Damage": "#ffffff",
    "Low damage": "#ffffff",
    "No damage": "#ffffff",
}

# Create the interactive map
trace_points = go.Scattermapbox(
    lat=plot_gdf["lat"],
    lon=plot_gdf["lon"],
    mode="markers",
    marker=dict(
        size=8,
        color=plot_gdf["DmgCls_4"].map(color_map).fillna("#ffffff"),
        opacity=0.95,
    ),
    text=plot_gdf["DmgCls_4"],
    hovertemplate="<b>%{text}</b><extra></extra>",
    name="Damage class",
)

fig = go.Figure(data=[trace_points])
fig.update_layout(
    mapbox=dict(
        style="mapbox://styles/mapbox/light-v10",
        center=dict(lon=37.16, lat=36.20),
        zoom=10,
        accesstoken=MAPBOX_TOKEN,
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    height=850,
    showlegend=False,
)
fig.show()

In [ ]:
from pathlib import Path
import networkx as nx
import pandas as pd

# Export the network graph as GEXF in the Downloads folder
output_dir = Path.home() / "Downloads"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "aleppo_network.gexf"

# Build the graph from the filtered Aleppo damage-building nodes
if "plot_gdf" in globals() and globals()["plot_gdf"] is not None:
    point_data = globals()["plot_gdf"].dropna(subset=["lon", "lat"]).copy()
    graph = nx.Graph()

    for idx, row in point_data.iterrows():
        neighborhood = (
            row.get("Neighbrhd") or row.get("StlmtNme") or row.get("Name") or "Unknown"
        )
        graph.add_node(
            idx,
            label=row.get("DmgCls_4", "building"),
            lon=float(row["lon"]),
            lat=float(row["lat"]),
            neighborhood=str(neighborhood),
            damage_class=str(row.get("DmgCls_4", "")),
        )

    # Connect nearby filtered nodes within the same neighborhood
    for neighborhood, nodes in (
        point_data.groupby("Neighbrhd") if "Neighbrhd" in point_data.columns else point_data.groupby(point_data.index)
    ):
        node_ids = [node for node in nodes.index if node in graph.nodes]
        if len(node_ids) < 2:
            continue

        ordered_nodes = sorted(
            node_ids,
            key=lambda n: (
                graph.nodes[n].get("lon", 0),
                graph.nodes[n].get("lat", 0),
                str(n),
            ),
        )
        for i in range(len(ordered_nodes) - 1):
            if not graph.has_edge(ordered_nodes[i], ordered_nodes[i + 1]):
                graph.add_edge(ordered_nodes[i], ordered_nodes[i + 1], weight=1)

    nx.write_gexf(graph, str(output_path))
    print(
        f"Saved GEXF to {output_path} ({graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges)"
    )
else:
    print("No filtered point data named plot_gdf was found in the notebook.")


Saved GEXF to /Users/khaledalanjery/Downloads/aleppo_network.gexf (1500 nodes, 260227 edges)


In [ ]:
from pathlib import Path
import json

# Export the filtered Aleppo damage points to JSON in the Downloads folder
output_dir = Path.home() / "Downloads"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "aleppo_damage_points.json"

if "plot_gdf" in globals():
    json_data = plot_gdf[["DmgCls_4", "lon", "lat"]].to_dict(orient="records")
    output_path.write_text(json.dumps(json_data, indent=2))
    print(f"Saved JSON to {output_path}")
else:
    print("No GeoDataFrame named plot_gdf was found in the notebook.")

Saved JSON to /Users/khaledalanjery/Downloads/aleppo_damage_points.json


In [11]:
from pathlib import Path

# Export the filtered Aleppo damage points to GeoJSON in the Downloads folder
output_dir = Path.home() / "Downloads"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "aleppo_damage_points.geojson"

if "plot_gdf" in globals():
    plot_gdf[["DmgCls_4", "geometry"]].to_file(output_path, driver="GeoJSON")
    print(f"Saved GeoJSON to {output_path}")
else:
    print("No GeoDataFrame named plot_gdf was found in the notebook.")

Saved GeoJSON to /Users/khaledalanjery/Downloads/aleppo_damage_points.geojson
